# PMD Maintainability — Nesting Depth Raw Output Extraction (Java)

This notebook analyzes **Java repositories** with **PMD** and captures the complete raw tool output for maintainability nesting-depth metric derivation and validation.

**Default benchmark repository:** [spring-projects/spring-framework](https://github.com/spring-projects/spring-framework)

The notebook supports:
- **Mode 1:** Clone from a Git repository URL
- **Mode 2:** Analyze an already-cloned local repository path

All deliverables are written to the configured `OUTPUT_DIR`.

**Prerequisite:** Java (JRE/JDK 11+) must be available on `PATH` for PMD execution.

## Section 1 — Install Dependencies

Install open-source Python packages and download the open-source PMD distribution automatically.

In [ ]:
!pip install -q pandas gitpython jupyter requests tqdm

In [ ]:
import io
import os
import shutil
import zipfile
from pathlib import Path
from urllib.request import urlopen

# Prevent cloned repositories from shadowing Python stdlib via PYTHONPATH
os.environ.pop("PYTHONPATH", None)


def download_pmd_distribution(
    pmd_version: str = "7.0.0",
    install_root: Path = Path("."),
) -> Path:
    pmd_home = install_root / f"pmd-bin-{pmd_version}"
    if (pmd_home / "bin").exists():
        print(f"PMD already installed at: {pmd_home}")
        return pmd_home.resolve()

    url = (
        f"https://github.com/pmd/pmd/releases/download/"
        f"pmd_releases%2F{pmd_version}/pmd-dist-{pmd_version}-bin.zip"
    )
    zip_path = install_root / f"pmd-dist-{pmd_version}-bin.zip"
    install_root.mkdir(parents=True, exist_ok=True)

    print(f"Downloading PMD {pmd_version} from GitHub releases...")
    with urlopen(url, timeout=120) as response, open(zip_path, "wb") as handle:
        total = int(response.headers.get("Content-Length", 0) or 0)
        downloaded = 0
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)
            downloaded += len(chunk)
            if total:
                print(f"Downloaded {downloaded / total:.1%}", end="\r")

    print(f"\nExtracting {zip_path.name}...")
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(install_root)

    if not (pmd_home / "bin").exists():
        raise RuntimeError(f"PMD extraction failed; expected directory: {pmd_home}")

    print(f"PMD installed at: {pmd_home.resolve()}")
    return pmd_home.resolve()


PMD_HOME = download_pmd_distribution(pmd_version="7.0.0", install_root=Path("../runtimes"))

## Section 2 — Configuration

Set execution mode, repository source, output location, and PMD ruleset options.

- Set `USE_GIT_URL = True` to clone from `REPO_URL`.
- Set `USE_GIT_URL = False` to analyze `LOCAL_REPO_PATH` directly.
- When cloning, use `IF_CLONE_EXISTS` to choose between reusing or re-cloning an existing local copy.

In [ ]:
USE_GIT_URL = True

REPO_URL = "https://github.com/spring-projects/spring-framework.git"

LOCAL_REPO_PATH = "/content/spring-framework"

OUTPUT_DIR = "./outputs"

PMD_HOME = "../runtimes/pmd-bin-7.0.0"

JDK_HOME = "../runtimes/jdk-21"

RULESET = "category/java/design.xml/AvoidDeeplyNestedIfStmts"

NESTING_RULE_NAME = "AvoidDeeplyNestedIfStmts"

IF_CLONE_EXISTS = "reuse"

CLONE_DEPTH = 1

WORKSPACE_DIR = "./workspace"

STREAM_RAW_OUTPUT = True

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark (100% predictable nesting outcomes):
# USE_GIT_URL = False
# LOCAL_REPO_PATH = "./workspace/java_nesting_benchmark"

## Utility Functions

Modular helpers for logging, repository setup, Java file discovery, PMD execution, and nesting-depth extraction.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Prevent cloned repositories from shadowing Python stdlib via PYTHONPATH
os.environ.pop("PYTHONPATH", None)
sys.path = [
    path for path in sys.path
    if "site-packages" in path.replace("\\", "/") or path in {"", "."}
]

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError


EXCLUDED_DIR_NAMES = {
    ".git",
    "target",
    "build",
    "out",
    "bin",
    "generated",
    "node_modules",
    ".gradle",
    ".idea",
}

JAVA_EXTENSION = ".java"

PMD_TEXT_VIOLATION_PATTERN = re.compile(
    r"^(?P<file>.+):(?P<line>\d+):\s*(?P<rule>[^:]+):\s*(?P<message>.*)$"
)

DEPTH_IN_MESSAGE_PATTERN = re.compile(
    r"(?:depth|nested|nesting)[^0-9]*(\d+)", re.IGNORECASE
)

INTEGER_PATTERN = re.compile(r"\b(\d+)\b")


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.error_log_path.exists():
            self.error_log_path.write_text("", encoding="utf-8")

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        with self.error_log_path.open("a", encoding="utf-8") as handle:
            handle.write(line)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)

    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')

    logger.info(f"Cloning {repo_url} into {clone_path} (depth={clone_depth})")
    try:
        clone_kwargs = {"depth": clone_depth} if clone_depth else {}
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Clone failed for {repo_url}: {exc}")
        raise

    logger.info(f"Clone completed: {clone_path}")
    return clone_path.resolve()


def validate_local_repository(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        message = f"Local repository path does not exist: {local_repo_path}"
        logger.error(message)
        raise FileNotFoundError(message)

    if not local_repo_path.is_dir():
        message = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(message)
        raise NotADirectoryError(message)

    has_git = (local_repo_path / ".git").exists()
    has_java = any(local_repo_path.rglob("*.java"))

    if not has_git and not has_java:
        message = (
            f"Path is neither a Git repository nor a Java source directory: {local_repo_path}"
        )
        logger.error(message)
        raise ValueError(message)

    if has_git:
        try:
            Repo(local_repo_path)
        except InvalidGitRepositoryError as exc:
            logger.error(f"Invalid Git repository at {local_repo_path}: {exc}")
            raise

    logger.info(f"Validated local repository path: {local_repo_path}")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool,
    repo_url: str,
    local_repo_path: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        logger.info("Execution mode: Git repository URL")
        return clone_or_reuse_repository(
            repo_url, workspace_dir, if_clone_exists, logger, clone_depth=clone_depth
        )
    logger.info("Execution mode: Local repository path")
    return validate_local_repository(Path(local_repo_path), logger)


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def discover_java_files(repo_path: Path) -> list[Path]:
    java_files: list[Path] = []
    for file_path in repo_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() != JAVA_EXTENSION:
            continue
        if should_exclude_path(file_path.relative_to(repo_path)):
            continue
        java_files.append(file_path.resolve())
    java_files.sort()
    return java_files


def compute_repository_stats(repo_path: Path, java_files: list[Path]) -> dict[str, Any]:
    total_size_bytes = sum(file_path.stat().st_size for file_path in java_files)
    directory_count = sum(
        1
        for current_path, _, _ in os.walk(repo_path)
        if not should_exclude_path(Path(current_path).relative_to(repo_path))
    )
    return {
        "java_file_count": len(java_files),
        "repository_size_bytes": total_size_bytes,
        "directory_count": directory_count,
    }


def save_java_file_list(java_files: list[Path], repo_path: Path, output_csv: Path) -> None:
    rows = [
        {
            "absolute_path": str(file_path),
            "relative_path": str(file_path.relative_to(repo_path)),
        }
        for file_path in java_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def resolve_pmd_executable(pmd_home: Path) -> Path:
    pmd_home = pmd_home.resolve()
    if sys.platform.startswith("win"):
        candidate = pmd_home / "bin" / "pmd.bat"
    else:
        candidate = pmd_home / "bin" / "pmd"
    if not candidate.exists():
        raise FileNotFoundError(f"PMD executable not found: {candidate}")
    return candidate


def configure_java_runtime(jdk_home: Path, logger: NotebookLogger) -> None:
    jdk_home = jdk_home.resolve()
    java_bin = jdk_home / "bin"
    java_exe = java_bin / ("java.exe" if sys.platform.startswith("win") else "java")
    if java_exe.exists():
        os.environ["JAVA_HOME"] = str(jdk_home)
        os.environ["PATH"] = str(java_bin) + os.pathsep + os.environ.get("PATH", "")
        logger.info(f"Configured shared JDK: {jdk_home}")
    else:
        logger.info(f"Shared JDK not found at {jdk_home}; using system Java if available.")


def verify_java_runtime(logger: NotebookLogger) -> None:
    try:
        completed = subprocess.run(
            ["java", "-version"],
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            check=False,
        )
    except FileNotFoundError as exc:
        logger.error("Java runtime not found. Install JDK/JRE 11+ and ensure 'java' is on PATH.")
        raise RuntimeError("Java runtime is required for PMD execution.") from exc

    if completed.returncode != 0 and not completed.stderr.strip():
        logger.error(f"Unable to verify Java runtime: {completed.stderr}")
        raise RuntimeError("Java runtime verification failed.")


def build_pmd_command(
    pmd_executable: Path,
    repo_path: Path,
    ruleset: str,
    output_format: str,
) -> list[str]:
    return [
        str(pmd_executable),
        "check",
        "-d",
        str(repo_path),
        "-R",
        ruleset,
        "-f",
        output_format,
        "--no-cache",
        "--no-progress",
    ]


def run_pmd_command(
    command: list[str],
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> tuple[str, str, int, bool]:
    try:
        if stream_raw:
            process = subprocess.Popen(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace",
                env=os.environ.copy(),
            )
            stdout_lines: list[str] = []
            stderr_lines: list[str] = []

            def _read_stream(pipe, sink, label):
                for line in iter(pipe.readline, ""):
                    print(line, end="", file=sys.stderr if label == "stderr" else sys.stdout)
                    sink.append(line)
                pipe.close()

            assert process.stdout is not None
            assert process.stderr is not None
            stdout_thread = threading.Thread(
                target=_read_stream, args=(process.stdout, stdout_lines, "stdout"), daemon=True
            )
            stderr_thread = threading.Thread(
                target=_read_stream, args=(process.stderr, stderr_lines, "stderr"), daemon=True
            )
            stdout_thread.start()
            stderr_thread.start()
            return_code = process.wait()
            stdout_thread.join()
            stderr_thread.join()
            stdout = "".join(stdout_lines)
            stderr = "".join(stderr_lines)
        else:
            completed = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                check=False,
                env=os.environ.copy(),
            )
            stdout = completed.stdout
            stderr = completed.stderr
            return_code = completed.returncode

        # PMD returns 4 when violations are found; that is a successful run.
        success = return_code in (0, 4)
        if not success:
            logger.error(
                f"PMD command failed with exit code {return_code}: {' '.join(command)}"
            )
        return stdout, stderr, return_code, success
    except Exception as exc:
        logger.error(f"PMD execution exception: {exc}")
        return "", str(exc), -1, False


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw_output = stdout
    if stderr:
        if raw_output and not raw_output.endswith("\n"):
            raw_output += "\n"
        raw_output += stderr
    return raw_output


def run_pmd_suite(
    pmd_home: Path,
    repo_path: Path,
    ruleset: str,
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> dict[str, str]:
    verify_java_runtime(logger)
    pmd_executable = resolve_pmd_executable(pmd_home)
    logger.info(f"Running PMD on repository: {repo_path}")

    text_command = build_pmd_command(pmd_executable, repo_path, ruleset, "text")
    text_stdout, text_stderr, _, text_ok = run_pmd_command(
        text_command, logger, stream_raw=stream_raw
    )

    csv_command = build_pmd_command(pmd_executable, repo_path, ruleset, "csv")
    csv_stdout, csv_stderr, _, csv_ok = run_pmd_command(csv_command, logger, stream_raw=False)
    if csv_stderr.strip():
        logger.error(f"PMD CSV stderr: {csv_stderr.strip()}")

    xml_command = build_pmd_command(pmd_executable, repo_path, ruleset, "xml")
    xml_stdout, xml_stderr, _, xml_ok = run_pmd_command(xml_command, logger, stream_raw=False)
    if xml_stderr.strip():
        logger.error(f"PMD XML stderr: {xml_stderr.strip()}")

    return {
        "raw_text": combine_raw_streams(text_stdout, text_stderr),
        "csv_text": csv_stdout,
        "xml_text": xml_stdout,
        "execution_ok": str(text_ok or csv_ok or xml_ok),
    }


def parse_pmd_csv(csv_text: str) -> pd.DataFrame:
    if not csv_text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(csv_text.strip()))


def normalize_pmd_columns(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    renamed = frame.copy()
    renamed.columns = [str(col).strip().lower().replace(" ", "_") for col in renamed.columns]
    return renamed


def parse_pmd_text_violations(raw_text: str) -> pd.DataFrame:
    rows = []
    for line in raw_text.splitlines():
        match = PMD_TEXT_VIOLATION_PATTERN.match(line.strip())
        if not match:
            continue
        rows.append(
            {
                "file": match.group("file").strip(),
                "begin_line": int(match.group("line")),
                "end_line": int(match.group("line")),
                "rule": match.group("rule").strip(),
                "priority": "",
                "message": match.group("message").strip(),
            }
        )
    return pd.DataFrame(rows)


def parse_pmd_xml_violations(xml_text: str) -> pd.DataFrame:
    if not xml_text.strip():
        return pd.DataFrame()
    rows = []
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        return pd.DataFrame()

    for file_node in root.iter():
        if not str(file_node.tag).endswith("file"):
            continue
        file_name = file_node.attrib.get("name", "")
        for violation in file_node:
            if not str(violation.tag).endswith("violation"):
                continue
            begin_line = int(violation.attrib.get("beginline", violation.attrib.get("beginLine", 0)) or 0)
            end_line = int(violation.attrib.get("endline", violation.attrib.get("endLine", begin_line)) or begin_line)
            rows.append(
                {
                    "file": file_name,
                    "begin_line": begin_line,
                    "end_line": end_line,
                    "rule": violation.attrib.get("rule", ""),
                    "priority": violation.attrib.get("priority", ""),
                    "message": (violation.text or "").strip(),
                }
            )
    return pd.DataFrame(rows)


def violations_from_csv(csv_df: pd.DataFrame) -> pd.DataFrame:
    if csv_df.empty:
        return pd.DataFrame()
    frame = normalize_pmd_columns(csv_df)
    rows = []
    for _, record in frame.iterrows():
        rule = str(record.get("rule", ""))
        file_name = str(record.get("file", record.get("filename", "")))
        line_value = record.get("line", record.get("beginline", record.get("begin_line", "")))
        message = str(record.get("description", record.get("message", "")))
        priority = record.get("priority", "")
        rows.append(
            {
                "file": file_name,
                "begin_line": line_value,
                "end_line": line_value,
                "rule": rule,
                "priority": priority,
                "message": message,
            }
        )
    return pd.DataFrame(rows)


def extract_depth_from_message(message: str) -> int | None:
    depth_match = DEPTH_IN_MESSAGE_PATTERN.search(message)
    if depth_match:
        return int(depth_match.group(1))
    numbers = [int(value) for value in INTEGER_PATTERN.findall(message)]
    return numbers[-1] if numbers else None


def strip_java_comments_and_strings(line: str) -> str:
    line = re.sub(r"//.*$", "", line)
    line = re.sub(r"/\*.*?\*/", "", line)
    line = re.sub(r'"(\\.|[^"\\])*"', '""', line)
    line = re.sub(r"'(\\.|[^'\\])*'", "''", line)
    return line


def derive_if_nesting_depth_at_line(source_text: str, target_line: int) -> int:
    lines = source_text.splitlines()
    if target_line < 1 or target_line > len(lines):
        return 0

    max_depth = 0
    current_if_depth = 0
    brace_depth = 0
    if_pattern = re.compile(r"\bif\s*\(")

    for index, line in enumerate(lines[:target_line], start=1):
        code = strip_java_comments_and_strings(line)
        if if_pattern.search(code):
            current_if_depth += 1
            max_depth = max(max_depth, current_if_depth)
        for char in code:
            if char == "{":
                brace_depth += 1
            elif char == "}":
                brace_depth = max(brace_depth - 1, 0)
                if brace_depth == 0:
                    current_if_depth = 0

    return int(max_depth)


def build_nesting_depth_findings(
    violations_df: pd.DataFrame,
    nesting_rule_name: str,
    logger: NotebookLogger,
) -> tuple[pd.DataFrame, int]:
    if violations_df.empty:
        return pd.DataFrame(
            columns=[
                "file",
                "begin_line",
                "end_line",
                "rule",
                "priority",
                "message",
                "detected_nesting_depth",
                "status",
            ]
        ), 0

    nesting_df = violations_df[
        violations_df["rule"].astype(str).str.contains(nesting_rule_name, case=False, na=False)
    ].copy()

    rows = []
    files_failed = 0
    source_cache: dict[str, str] = {}

    for _, record in nesting_df.iterrows():
        file_path = str(record.get("file", "")).strip()
        begin_line = int(record.get("begin_line", 0) or 0)
        end_line = int(record.get("end_line", begin_line) or begin_line)
        message = str(record.get("message", "")).strip()
        reported_depth = extract_depth_from_message(message)

        if reported_depth is not None:
            rows.append(
                {
                    "file": file_path,
                    "begin_line": begin_line,
                    "end_line": end_line,
                    "rule": record.get("rule", ""),
                    "priority": record.get("priority", ""),
                    "message": message,
                    "detected_nesting_depth": reported_depth,
                    "status": "reported",
                }
            )
            continue

        try:
            if file_path not in source_cache:
                source_cache[file_path] = Path(file_path).read_text(encoding="utf-8", errors="replace")
            source = source_cache[file_path]
            if_pattern = re.compile(r"\bif\s*\(")
            snippet = source.splitlines()[begin_line - 1 : end_line]
            derived_depth = sum(
                1 for line in snippet if if_pattern.search(strip_java_comments_and_strings(line))
            )
            rows.append(
                {
                    "file": file_path,
                    "begin_line": begin_line,
                    "end_line": end_line,
                    "rule": record.get("rule", ""),
                    "priority": record.get("priority", ""),
                    "message": message,
                    "detected_nesting_depth": derived_depth,
                    "status": "derived",
                }
            )
        except Exception as exc:
            files_failed += 1
            logger.error(f"Failed to derive nesting depth for {file_path}:{begin_line}: {exc}")
            rows.append(
                {
                    "file": file_path,
                    "begin_line": begin_line,
                    "end_line": end_line,
                    "rule": record.get("rule", ""),
                    "priority": record.get("priority", ""),
                    "message": message,
                    "detected_nesting_depth": None,
                    "status": "failed",
                }
            )

    return pd.DataFrame(rows), files_failed


def merge_violation_sources(*frames: pd.DataFrame) -> pd.DataFrame:
    valid_frames = [frame for frame in frames if frame is not None and not frame.empty]
    if not valid_frames:
        return pd.DataFrame()
    combined = pd.concat(valid_frames, ignore_index=True)
    dedupe_cols = ["file", "begin_line", "rule", "message"]
    for col in dedupe_cols:
        if col not in combined.columns:
            combined[col] = ""
    return combined.drop_duplicates(subset=dedupe_cols, keep="first").reset_index(drop=True)


def compute_maintainability_summary(findings_df: pd.DataFrame) -> pd.DataFrame:
    valid = findings_df[findings_df["detected_nesting_depth"].notna()].copy()
    if valid.empty:
        return pd.DataFrame(
            [
                {"metric_name": "Maintainability_Nesting_Depth", "metric_value": 0},
                {"metric_name": "Average_Nesting_Depth", "metric_value": 0},
            ]
        )
    max_depth = int(valid["detected_nesting_depth"].max())
    avg_depth = float(valid["detected_nesting_depth"].mean())
    return pd.DataFrame(
        [
            {"metric_name": "Maintainability_Nesting_Depth", "metric_value": max_depth},
            {"metric_name": "Average_Nesting_Depth", "metric_value": round(avg_depth, 4)},
        ]
    )


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW PMD OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")

## Section 3 — Repository Setup

Resolve the repository path based on configuration, initialize output directories, and verify PMD installation.

In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
PMD_HOME_PATH = Path(PMD_HOME).resolve()
JDK_HOME_PATH = Path(JDK_HOME).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / "error_log.txt"

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    configure_java_runtime(JDK_HOME_PATH, logger)
    resolve_pmd_executable(PMD_HOME_PATH)
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f"Repository setup failed: {exc}")
    raise

logger.info(f"Repository ready at: {REPO_PATH}")
logger.info(f"PMD home: {PMD_HOME_PATH}")

## Section 4 — Discover Java Files

Recursively discover `.java` files while excluding build and tooling directories.

In [ ]:
JAVA_FILES = discover_java_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, JAVA_FILES)

JAVA_FILES_CSV = OUTPUT_PATH / "java_files.csv"
save_java_file_list(JAVA_FILES, REPO_PATH, JAVA_FILES_CSV)

print(f"Total Java Files Found: {len(JAVA_FILES)}")
print(f"Repository Size (Java files only): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Total Directories (excluding filtered paths): {REPO_STATS['directory_count']:,}")
print(f"Saved file list to: {JAVA_FILES_CSV}")

## Section 5 — Execute PMD

Run PMD against the repository using the configured nesting-depth ruleset.

Example equivalent command:

```bash
../runtimes/pmd-bin-7.0.0/bin/pmd check \
  -d <repo_path> \
  -R category/java/design.xml/AvoidDeeplyNestedIfStmts \
  -f text
```

Raw stdout and stderr are preserved without suppression.

In [ ]:
if not JAVA_FILES:
    logger.error("No Java files discovered; skipping PMD execution.")
    PMD_OUTPUTS = {"raw_text": "", "csv_text": "", "xml_text": "", "execution_ok": "False"}
    FILES_SUCCESS = 0
    FILES_FAILED = len(JAVA_FILES)
else:
    PMD_OUTPUTS = run_pmd_suite(
        pmd_home=PMD_HOME_PATH,
        repo_path=REPO_PATH,
        ruleset=RULESET,
        logger=logger,
        stream_raw=STREAM_RAW_OUTPUT,
    )
    FILES_SUCCESS = len(JAVA_FILES)
    FILES_FAILED = 0

logger.info(f"PMD execution complete. Raw output size: {len(PMD_OUTPUTS['raw_text']):,} characters")

## Section 6 — Raw Output Extraction

Persist complete raw PMD text output, CSV output, and XML output exactly as emitted by the tool.

In [ ]:
RAW_OUTPUT_PATH = OUTPUT_PATH / "pmd_raw_output.txt"
CSV_OUTPUT_PATH = OUTPUT_PATH / "pmd_output.csv"
XML_OUTPUT_PATH = OUTPUT_PATH / "pmd_output.xml"

raw_text_output = PMD_OUTPUTS["raw_text"]
RAW_OUTPUT_PATH.write_text(raw_text_output, encoding="utf-8")
CSV_OUTPUT_PATH.write_text(PMD_OUTPUTS["csv_text"], encoding="utf-8")
XML_OUTPUT_PATH.write_text(PMD_OUTPUTS["xml_text"], encoding="utf-8")

PMD_CSV_DF = parse_pmd_csv(PMD_OUTPUTS["csv_text"])
PMD_TEXT_VIOLATIONS_DF = parse_pmd_text_violations(raw_text_output)
PMD_XML_VIOLATIONS_DF = parse_pmd_xml_violations(PMD_OUTPUTS["xml_text"])
ALL_VIOLATIONS_DF = merge_violation_sources(
    violations_from_csv(PMD_CSV_DF),
    PMD_TEXT_VIOLATIONS_DF,
    PMD_XML_VIOLATIONS_DF,
)

logger.info(f"Saved raw output: {RAW_OUTPUT_PATH}")
logger.info(f"Saved CSV output: {CSV_OUTPUT_PATH}")
logger.info(f"Saved XML output: {XML_OUTPUT_PATH}")
logger.info(f"Total PMD findings: {len(ALL_VIOLATIONS_DF)}")

preview_raw_output(raw_text_output, RAW_OUTPUT_PREVIEW_LINES, RAW_OUTPUT_PATH)

## Section 7 — Maintainability Nesting Depth Extraction

Extract all violations produced by `AvoidDeeplyNestedIfStmts`.

If PMD does not report the actual nesting depth in the message, derive it by parsing the Java control-structure hierarchy at the violation line.

In [ ]:
NESTING_FINDINGS_DF, DERIVE_FAILURES = build_nesting_depth_findings(
    violations_df=ALL_VIOLATIONS_DF,
    nesting_rule_name=NESTING_RULE_NAME,
    logger=logger,
)

NESTING_FINDINGS_CSV = OUTPUT_PATH / "nesting_depth_findings.csv"
NESTING_FINDINGS_DF.to_csv(NESTING_FINDINGS_CSV, index=False)
FILES_FAILED += DERIVE_FAILURES

logger.info(f"Saved nesting depth findings: {NESTING_FINDINGS_CSV}")
logger.info(f"Nesting depth findings count: {len(NESTING_FINDINGS_DF)}")

if not NESTING_FINDINGS_DF.empty:
    display(NESTING_FINDINGS_DF.head(10))
else:
    print("No nesting depth findings detected.")

## Section 8 — Metric Computation

Compute repository-level maintainability nesting depth metrics:

- **Maintainability_Nesting_Depth** = `max(detected_nesting_depth)`
- **Average_Nesting_Depth** = `Σ detected_nesting_depth / total_violating_methods`

In [ ]:
SUMMARY_DF = compute_maintainability_summary(NESTING_FINDINGS_DF)
SUMMARY_CSV = OUTPUT_PATH / "maintainability_nesting_depth_summary.csv"
SUMMARY_DF.to_csv(SUMMARY_CSV, index=False)

logger.info(f"Saved maintainability summary: {SUMMARY_CSV}")
display(SUMMARY_DF)

## Section 9 — Summary Dashboard

Overview of analysis coverage, PMD findings, and nesting-depth metrics.

In [ ]:
max_depth = SUMMARY_DF.loc[
    SUMMARY_DF["metric_name"] == "Maintainability_Nesting_Depth", "metric_value"
].iloc[0]
avg_depth = SUMMARY_DF.loc[
    SUMMARY_DF["metric_name"] == "Average_Nesting_Depth", "metric_value"
].iloc[0]

summary_df = pd.DataFrame(
    [
        {"Metric": "Total Java Files", "Value": len(JAVA_FILES)},
        {"Metric": "Files Successfully Analyzed", "Value": FILES_SUCCESS},
        {"Metric": "Files Failed", "Value": FILES_FAILED},
        {"Metric": "Total PMD Findings", "Value": len(ALL_VIOLATIONS_DF)},
        {"Metric": "Nesting Depth Findings", "Value": len(NESTING_FINDINGS_DF)},
        {"Metric": "Maximum Nesting Depth", "Value": max_depth},
        {"Metric": "Average Nesting Depth", "Value": avg_depth},
    ]
)

display(summary_df)

deliverables = [
    RAW_OUTPUT_PATH,
    CSV_OUTPUT_PATH,
    XML_OUTPUT_PATH,
    JAVA_FILES_CSV,
    NESTING_FINDINGS_CSV,
    SUMMARY_CSV,
    ERROR_LOG_PATH,
]

print("\nDeliverables:")
for deliverable in deliverables:
    status = "OK" if deliverable.exists() else "MISSING"
    print(f"  [{status}] {deliverable}")

## Section 10 — Error Handling

Failures encountered during cloning, validation, or PMD execution are appended to `outputs/error_log.txt`.

In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding="utf-8"))
else:
    print("No errors logged.")

## Section 11 — Deliverables

Upon successful completion, the following artifacts are available under `outputs/`:

```text
outputs/
├── pmd_raw_output.txt
├── pmd_output.csv
├── pmd_output.xml
├── java_files.csv
├── nesting_depth_findings.csv
├── maintainability_nesting_depth_summary.csv
└── error_log.txt
```

The notebook is designed to run end-to-end in Jupyter Notebook and Google Colab without manual intervention.